# Cross-Region Generalization Test: Tuktoyaktuk-Trained Model on Cambridge Bay

Runs `09`'s already-trained checkpoint (`s1_tuk_pcrtc_realattrs_spatialsplit_unet_best.pth`
-- the best leakage-free Tuktoyaktuk model) on Cambridge Bay's 2101
newly-extracted Sentinel-1/LiDAR patches. **No retraining, no train/val
split** -- every Cambridge Bay patch is held-out test data. This answers
the actual question Michel asked: does what the model learned on
Tuktoyaktuk transfer to a region it has never seen?

**Confound to keep in mind when interpreting the result**: Cambridge
Bay's nearest available Sentinel-1 imagery is from May 2025, ~402-409
days after the LiDAR survey date and a different season (post-thaw
summer vs. the April/frozen-season survey) -- documented in `CONCEPTS.md`
and `11_cambridge_bay_patch_extraction.ipynb`. Any result here reflects
region generalization *and* this single season/date confound, not region
generalization alone. Still a single, clean confound -- not the
three-way one (region + date + polarization) that ruled out Pond Inlet.

## GPU configuration

In [ ]:
# Require CUDA -- this notebook only does inference, but still needs the GPU for the diffusion sampler
import os
import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and configuration

In [ ]:
# Region-specific paths -- checkpoint comes from 09 (Tuktoyaktuk), data comes from Cambridge Bay
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
CHECKPOINT_NAME = 's1_tuk_pcrtc_realattrs_spatialsplit_unet_best.pth'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_cambridge_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
TIMESTEPS = 1000
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
CAMBRIDGE_BAY_SURVEY_DATE = __import__('datetime').date(2024, 4, 18)

print('LIDAR_DIR:', LIDAR_DIR)
print('S1_DIR:', S1_DIR)
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)

## Import Tessa's baseline implementation

In [ ]:
# Import the model, scheduler, sampler, and metric functions from Tessa's baseline repo
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

## Dataset adapter -- identical to `09`'s real-attrs class, just no train/val split

Every patch is test data here, so there's no split logic at all --
`CambridgeBayS1Dataset` simply loads every matched patch.

In [ ]:
# Real-attrs dataset adapter, unchanged from 09 except the survey date used for age_norm
def build_real_attrs(s1_path, times, context_k, survey_date):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])  # 't0' -> 0
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - survey_date).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class CambridgeBayS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), survey_date=None):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.survey_date = survey_date

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k, self.survey_date)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Build the test set -- every matched Cambridge Bay patch, no split

Uses whichever patch IDs have both a LiDAR patch and a matched
Sentinel-1 directory (should be all 2101 from `11`'s matching step).

In [ ]:
# All matched Cambridge Bay patches become the test set -- no train/val split, nothing here is used for training
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
test_ids = sorted(lidar_ids & s1_ids)
assert test_ids, 'No paired Cambridge Bay Sentinel-1/LiDAR patches found.'
print(f'Cambridge Bay test set: {len(test_ids)} patches (no split -- all held out for inference)')

test_dataset = CambridgeBayS1Dataset(S1_DIR, LIDAR_DIR, test_ids, CONTEXT_K, TARGET_HW, survey_date=CAMBRIDGE_BAY_SURVEY_DATE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Load the Tuktoyaktuk-trained checkpoint (frozen, no further training)

In [ ]:
# Initialize the same architecture 09 used, then load its trained weights -- frozen, eval-only from here on
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else None

checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.6f}')
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Run inference and compute reconstruction metrics

Identical metric suite to every other pcrtc evaluation in this project,
for direct comparison against Tuktoyaktuk's in-region numbers.

In [ ]:
# Run DDIM inference on every Cambridge Bay patch and compute the same reconstruction metrics used throughout this project
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []
N_EXAMPLES = 6
with torch.no_grad():
    for batch in test_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            row = {
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            }
            metric_rows.append(row)
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id,
                    'gt': gt_i.squeeze().cpu().numpy(),
                    'pred': pred_i.squeeze().cpu().numpy(),
                    'mask': mask_i.squeeze().cpu().numpy(),
                })

metrics_path = OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_metrics.json'
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'})

## Comparison protocol

Compare this Cambridge Bay cross-region result against:
1. **`09`'s in-region Tuktoyaktuk validation** (`s1_pcrtc_realattrs_spatialsplit_validation_metrics.json`)
   -- the same model, tested on data from the region it was trained on.
   The gap between the two numbers reflects region generalization
   *combined with* the ~402-409 day / cross-season confound documented
   above, not region generalization in isolation.
2. **`08`'s unseen-date Tuktoyaktuk test** (same region, new date) -- if
   Cambridge Bay's drop is similar in size to `08`'s (ZNCC 0.52 to 0.01),
   that suggests the date/season confound dominates over anything
   region-specific. If Cambridge Bay holds up much better than `08`,
   that's a more encouraging sign for genuine spatial transfer despite
   the imperfect date match.

In [ ]:
# Compare against 09's in-region Tuktoyaktuk numbers to see how much the cross-region+cross-season test degrades performance
tuk_inregion_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'
new_mean = {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'}

if tuk_inregion_path.exists():
    tuk_rows = json.load(open(tuk_inregion_path))
    tuk_mean = {k: float(np.nanmean([r[k] for r in tuk_rows])) for k in tuk_rows[0] if k != 'patch_id'}
    print(f'{"metric":<20}{"Tuktoyaktuk in-region (09)":>28}{"Cambridge Bay cross-region":>28}')
    for k in new_mean:
        if k in tuk_mean:
            print(f'{k:<20}{tuk_mean[k]:>28.4f}{new_mean[k]:>28.4f}')
else:
    print('09 in-region metrics file not found -- compare manually against the documented 09 numbers.')